# Demo — Temperatura: ¿cuánta creatividad le pides al modelo?

> Demo que acompaña a [LLM e
> inferencia](../bloque1/01b-llms.qmd#sec-parametros-inferencia). Las
> probabilidades se midieron una vez con `Qwen3-0.6B` y se guardaron
> como datos estáticos, así que la demo se ejecuta en el navegador sin
> API ni clave.

Un LLM no «decide» la siguiente palabra: calcula una **probabilidad para
cada candidata** y saca una al azar. La **temperatura** controla cuánto
se respetan esas probabilidades. Con temperatura baja casi siempre gana
la favorita; con temperatura alta el abanico se aplana y las palabras
raras tienen opción.

Elige una frase, mueve la temperatura (o usa los botones *Determinista*,
*Equilibrada*, *Creativa* y *Caótica*) y pulsa **Generar 10**. Cada
palabra que sale suma en su barra (×n), y la raya negra de cada barra
marca la probabilidad original del modelo: así ves cuánto la ha
deformado la temperatura.

``` shinylive-python
#| '!! shinylive warning !!': |
#|   shinylive does not work in self-contained HTML documents.
#|   Please set `embed-resources: false` in your metadata.
#| standalone: true
#| viewerHeight: 1200

## file: app.py
import numpy as np
from shiny import reactive
from shiny.express import input, ui, render

from data import DATA

PROMPTS = DATA["prompts"]
BLUE, DARK, ORANGE, GREEN, GRAY = "#4B8BBE", "#306998", "#e65100", "#2e7d32", "#9e9e9e"
RARE = 0.02  # una palabra es «poco plausible» si el modelo le da menos del 2 %
MAX_SHOWN = 12

CUTS = {
    "none": "Sin recorte (todas las candidatas)",
    "k3": "Solo las 3 más probables (top-k = 3)",
    "p90": "Solo las que suman el 90 % (top-p = 0,9)",
}
PRESETS = [(0.0, "Determinista"), (0.7, "Equilibrada"), (1.2, "Creativa"), (2.0, "Caótica")]

ui.tags.style("""
body{font-family:system-ui,-apple-system,'Segoe UI',sans-serif}
.card{border:1px solid #e5e7eb;border-radius:10px;padding:10px 14px;margin:10px 0;background:#fff}
.muted{color:#6b7280;font-size:.85em}
.chip{display:inline-block;border:1px solid #306998;color:#306998;background:#fff;border-radius:999px;
  padding:2px 12px;margin:2px 6px 2px 0;font-size:.85em;cursor:pointer;user-select:none}
.chip:hover{background:#e8f0f8}.chip.on{background:#306998;color:#fff}
.fan{display:grid;grid-template-columns:96px 1fr 46px 34px;gap:3px 8px;align-items:center}
.fan .w{text-align:right}
.track{position:relative;height:16px;background:#f3f4f6;border-radius:4px}
.fill{position:absolute;left:0;top:0;bottom:0;border-radius:4px}
.orig{position:absolute;top:-3px;bottom:-3px;width:2px;background:#111;opacity:.55}
.count{font-size:.8em;font-weight:600;border-radius:999px;text-align:center;padding:0 4px}
.stats{display:flex;gap:10px;flex-wrap:wrap;margin:6px 0}
.stat{flex:1;min-width:110px;border:1px solid #e5e7eb;border-radius:8px;padding:4px 10px}
.stat b{font-size:1.4em;display:block}
.gen{display:inline-block;margin:3px 6px 3px 0;padding:3px 10px;border-radius:999px;background:#f3f4f6}
@keyframes pop{from{transform:scale(.6);opacity:0}to{transform:scale(1);opacity:1}}
.new{animation:pop .35s ease-out both}
""")

samples = reactive.value([])
last_batch = reactive.value(0)


def distribution(i, temp, cut):
    """Probabilidades tras aplicar temperatura y recorte, y máscara de las que sobreviven."""
    p = np.array(PROMPTS[i]["probs"], dtype=float)
    if temp <= 0:
        q = np.zeros_like(p)
        q[np.argmax(p)] = 1.0
    else:
        logits = np.log(np.maximum(p, 1e-12)) / temp
        q = np.exp(logits - logits.max())
        q /= q.sum()
    order = np.argsort(-q)
    keep = np.ones(len(q), dtype=bool)
    if cut == "k3":
        keep[order[3:]] = False
    elif cut == "p90":
        cum = np.cumsum(q[order])
        n = int(np.searchsorted(cum, 0.9) + 1)
        keep[order[n:]] = False
    q = np.where(keep, q, 0.0)
    return q / q.sum(), keep


def mood(temp):
    if temp <= 0:
        return "Determinista: siempre gana la favorita."
    if temp < 0.6:
        return "Conservadora: la favorita casi siempre gana."
    if temp <= 1.05:
        return "Cerca de 1: el abanico tal como lo calcula el modelo."
    if temp < 1.6:
        return "Creativa: las candidatas menos probables ganan peso."
    return "Muy plana: lo raro sale a menudo."


ui.input_select(
    "prompt", "Frase a continuar",
    {str(i): p["prompt"] + " …" for i, p in enumerate(PROMPTS)}, selected="0", width="100%",
)
ui.input_slider("temp", "Temperatura", 0, 2, 1, step=0.1, width="100%")


@render.ui
def presets():
    t = input.temp()
    chips = "".join(
        f"<span class='chip{' on' if abs(t - v) < 1e-9 else ''}' "
        f"onclick=\"Shiny.setInputValue('preset', {v}, {{priority: 'event'}})\">{name} · {v:g}</span>"
        for v, name in PRESETS
    )
    return ui.HTML(f"<div style='margin:-8px 0 6px'>{chips}</div><p class='muted'>{mood(t)}</p>")


ui.input_select("cut", "Recortar el abanico (opcional)", CUTS, selected="none", width="100%")

with ui.div(style="display:flex;gap:8px;flex-wrap:wrap;margin:4px 0 8px"):
    ui.input_action_button("gen1", "Generar 1", class_="btn-outline-primary")
    ui.input_action_button("gen10", "Generar 10", class_="btn-primary")
    ui.input_action_button("clear", "Borrar", class_="btn-outline-secondary")


@reactive.effect
@reactive.event(input.preset)
def _preset():
    ui.update_slider("temp", value=float(input.preset()))


@reactive.effect
@reactive.event(input.prompt, input.temp, input.cut, input.clear)
def _reset():
    samples.set([])
    last_batch.set(0)


def _draw(n):
    q, _ = distribution(int(input.prompt()), input.temp(), input.cut())
    draws = np.random.default_rng().choice(len(q), size=n, p=q)
    samples.set((samples() + [int(d) for d in draws])[-200:])
    last_batch.set(n)


@reactive.effect
@reactive.event(input.gen1)
def _gen1():
    _draw(1)


@reactive.effect
@reactive.event(input.gen10)
def _gen10():
    _draw(10)


@render.ui
def fan():
    i = int(input.prompt())
    words = PROMPTS[i]["words"]
    base = np.array(PROMPTS[i]["probs"], dtype=float)
    q, keep = distribution(i, input.temp(), input.cut())
    s = samples()
    counts = np.bincount(s, minlength=len(words)) if s else np.zeros(len(words), dtype=int)
    scale = max(q.max(), base.max())
    cells = []
    for j, w in enumerate(words):
        rare = base[j] < RARE
        color = ORANGE if rare else BLUE
        orig = f"<span class='orig' style='left:{100 * base[j] / scale:.1f}%'></span>"
        if keep[j]:
            fill = f"<span class='fill' style='width:{100 * q[j] / scale:.1f}%;background:{color}'></span>"
            label = f"{q[j]:.0%}" if q[j] >= 0.005 else "&lt;1 %"
            wstyle = f"color:{ORANGE}" if rare else ""
        else:
            fill, label = "", "fuera"
            wstyle = "text-decoration:line-through;opacity:.45"
        badge = (
            f"<span class='count' style='background:{color};color:#fff'>×{counts[j]}</span>"
            if counts[j] else "<span></span>"
        )
        cells.append(
            f"<span class='w' style='{wstyle}'>{w}</span>"
            f"<span class='track' style='opacity:{1 if keep[j] else .4}'>{fill}{orig}</span>"
            f"<span class='muted' style='font-size:.8em'>{label}</span>{badge}"
        )
    legend = (
        f"<p class='muted' style='margin-top:6px'>"
        f"<span style='color:{BLUE}'>■</span> plausible · "
        f"<span style='color:{ORANGE}'>■</span> poco plausible (el modelo le daba &lt;2 %) · "
        f"<b>│</b> probabilidad original (T = 1) · <b>×n</b> veces que ha salido</p>"
    )
    return ui.HTML(
        f"<div class='card'><b>Abanico de la siguiente palabra</b>"
        f"<div class='fan' style='margin-top:8px'>{''.join(cells)}</div>{legend}</div>"
    )


@render.ui
def results():
    i = int(input.prompt())
    s = samples()
    if not s:
        return ui.HTML(
            "<div class='card muted'>Pulsa «Generar 10» para ver qué escribe el modelo. "
            "Las veces que sale cada palabra se acumulan en el abanico.</div>"
        )
    words = PROMPTS[i]["words"]
    base = PROMPTS[i]["probs"]
    n_new = last_batch()
    shown = s[-MAX_SHOWN:][::-1]
    chips = "".join(
        f"<span class='gen{' new' if k < n_new else ''}' style='animation-delay:{0.04 * k:.2f}s'>"
        f"{PROMPTS[i]['prompt']} <b style='color:{ORANGE if base[d] < RARE else DARK}'>{words[d]}</b></span>"
        for k, d in enumerate(shown)
    )
    distinct = len(set(s))
    rare = sum(base[d] < RARE for d in s)
    stats = (
        f"<div class='stats'>"
        f"<div class='stat'><span class='muted'>Generadas</span><b>{len(s)}</b></div>"
        f"<div class='stat'><span class='muted'>Palabras distintas</span><b>{distinct}</b></div>"
        f"<div class='stat'><span class='muted'>Poco plausibles</span>"
        f"<b style='color:{ORANGE if rare else GREEN}'>{rare} <span style='font-size:.6em'>"
        f"({rare / len(s):.0%})</span></b></div></div>"
    )
    more = f"<p class='muted'>Se muestran las {MAX_SHOWN} últimas.</p>" if len(s) > MAX_SHOWN else ""
    return ui.HTML(f"<div class='card'><b>Lo que ha escrito el modelo</b>{stats}<div>{chips}</div>{more}</div>")


ui.input_checkbox("nums", "Ver los números", value=False)


@render.ui
def numbers():
    if not input.nums():
        return ui.HTML("")
    i = int(input.prompt())
    words = PROMPTS[i]["words"]
    base = PROMPTS[i]["probs"]
    q, _ = distribution(i, input.temp(), input.cut())
    rows = "".join(
        f"<tr><td>{w}</td><td style='text-align:right'>{base[j]:.1%}</td>"
        f"<td style='text-align:right'>{q[j]:.1%}</td></tr>"
        for j, w in enumerate(words)
    )
    return ui.HTML(
        "<table class='table table-sm' style='font-size:0.85em;max-width:360px'><thead><tr>"
        "<th>Palabra</th><th style='text-align:right'>Prob. original</th>"
        f"<th style='text-align:right'>Prob. ahora</th></tr></thead><tbody>{rows}</tbody></table>"
    )

## file: data.py
# DATA-START
DATA = {
 "model": "Qwen/Qwen3-0.6B",
 "prompts": [
  {
   "prompt": "El gato se subió al",
   "words": [
    "coche",
    "sofá",
    "árbol",
    "suelo",
    "camión",
    "muro",
    "balcón",
    "techo",
    "cielo",
    "armario",
    "tejado",
    "ordenador"
   ],
   "probs": [
    0.35132,
    0.23103,
    0.19831,
    0.05209,
    0.05153,
    0.03962,
    0.02643,
    0.02618,
    0.01643,
    0.00563,
    0.00108,
    0.00035
   ]
  },
  {
   "prompt": "Los fines de semana suelo ir al",
   "words": [
    "mar",
    "mercado",
    "centro",
    "trabajo",
    "campo",
    "cine",
    "bosque",
    "parque",
    "museo",
    "hospital",
    "teatro",
    "tribunal"
   ],
   "probs": [
    0.28835,
    0.27167,
    0.13232,
    0.11174,
    0.06082,
    0.05176,
    0.02716,
    0.01974,
    0.01741,
    0.01418,
    0.00459,
    0.00026
   ]
  },
  {
   "prompt": "Mi comida favorita es la",
   "words": [
    "pizza",
    "ensalada",
    "sopa",
    "pasta",
    "carne",
    "paella",
    "hamburguesa",
    "fruta",
    "tortilla",
    "lasaña",
    "democracia",
    "electricidad"
   ],
   "probs": [
    0.55052,
    0.25165,
    0.11013,
    0.04213,
    0.0194,
    0.01046,
    0.00711,
    0.00488,
    0.00359,
    8e-05,
    3e-05,
    2e-05
   ]
  },
  {
   "prompt": "Para el desayuno me gusta tomar un café con",
   "words": [
    "leche",
    "chocolate",
    "azúcar",
    "hielo",
    "mantequilla",
    "aceite",
    "sal",
    "nata",
    "canela",
    "pimienta",
    "tostadas",
    "miel"
   ],
   "probs": [
    0.89601,
    0.05219,
    0.02517,
    0.01793,
    0.00442,
    0.00259,
    0.00095,
    0.00054,
    0.00017,
    3e-05,
    0.0,
    0.0
   ]
  }
 ]
}
# DATA-END
```

## Qué observar

1.  Con la frase del **gato** y temperatura **0**, pulsa «Generar 10»
    varias veces: sale siempre la misma palabra. Es el modo
    determinista, el que se usa cuando quieres respuestas reproducibles.
2.  Sube a **1,0**. Ahora aparecen varias palabras, pero casi todas
    razonables. Es el abanico tal como lo calcula el modelo.
3.  Sube a **2,0**. El abanico se aplana: las barras naranjas (palabras
    que el modelo consideraba absurdas) empiezan a salir. Ese es el
    momento en que el texto deja de tener sentido.
4.  Cambia a la frase del **café**. El modelo está casi seguro de que
    sigue «leche», así que la temperatura apenas cambia nada hasta
    valores altos. La temperatura solo da libertad donde el modelo duda.
5.  Vuelve a 1,5 y elige **«Solo las 3 más probables»**. Las palabras
    raras desaparecen aunque la temperatura sea alta: recortar el
    abanico es la forma de tener variedad sin los disparates.

## Qué has visto

- El modelo no decide: **saca una palabra de un abanico de
  probabilidades**. La temperatura reparte esa probabilidad de forma más
  igualitaria (alta) o más concentrada (baja).
- A temperatura 0 siempre sale lo mismo. A temperaturas altas aparece lo
  absurdo. Para tareas factuales o estructuradas conviene temperatura
  baja; para escritura creativa, media.
- **Top-k** y **top-p** recortan el abanico antes de sortear, y por eso
  se combinan con la temperatura en producción.

> **Qué es real y qué está simplificado**
>
> - Las probabilidades son reales, de `Qwen3-0.6B`, un modelo pequeño:
>   sus preferencias (por ejemplo, que un gato se suba «al coche» antes
>   que «al sofá») son menos sensatas que las de un LLM grande.
> - Las candidatas son **doce palabras elegidas a mano**, incluidas
>   algunas absurdas para que se vea la cola. Un modelo real reparte
>   probabilidad entre decenas de miles de tokens, y su siguiente token
>   puede ser un fragmento de palabra.
> - Las probabilidades se **renormalizaron entre esas doce candidatas**,
>   no son las absolutas del modelo.
> - Una palabra de varios tokens se puntúa multiplicando las
>   probabilidades de sus tokens.

Para ver cómo se fijan estos parámetros al llamar a un modelo, vuelve a
[LLM e inferencia](../bloque1/01b-llms.qmd#sec-parametros-inferencia).